# Module 02: Unified Information-Theoretic Feature Selector

## Learning Objectives

By completing this notebook, you will:
1. Implement all five major ITFS criteria (JMI, CMIM, DISR, ICAP, mRMR) from the Brown et al. (2012) unified CLM framework
2. Apply all criteria to the same real dataset and compare selected feature sets
3. Quantify feature-set overlap using Jaccard similarity
4. Visualise how different criteria prioritise features differently
5. Evaluate downstream model accuracy for each criterion and understand when each wins

## Prerequisites
- Guide 01: The CLM unified framework (Brown et al., 2012)
- Module 01: Shannon entropy and mutual information basics
- Python: numpy, pandas, sklearn, matplotlib

## Estimated Time: 15 minutes

## Reference
Brown, G., Pocock, A., Zhao, M-J., Lujan, M. (2012). Conditional Likelihood Maximisation: A Unifying Framework for Information Theoretic Feature Selection. *JMLR* 13, 27–66.

---

## 1. Setup

We use the UCI Wine Quality dataset — a real regression/classification dataset with 11 physicochemical features and a quality score target. It has enough features to show meaningful differences between criteria without being computationally heavy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from itertools import combinations
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import KBinsDiscretizer, StandardScaler
from sklearn.metrics import mutual_info_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries loaded.")

### 1.1 Load Dataset

We use the breast cancer dataset (30 features, binary target) from scikit-learn. This dataset has known redundancy among features (many features are correlated) and some complementarity — making it an ideal testbed for comparing ITFS criteria.

In [ ]:
# Load the breast cancer dataset
# 30 physicochemical features, binary target (malignant/benign)
data = load_breast_cancer()
X_raw = data.data
y = data.target
feature_names = list(data.feature_names)

print(f"Dataset: {data.filename if hasattr(data, 'filename') else 'Breast Cancer Wisconsin'}")
print(f"Samples: {X_raw.shape[0]}")
print(f"Features: {X_raw.shape[1]}")
print(f"Target classes: {np.unique(y)} ({data.target_names})")
print(f"\nFirst 5 feature names:")
for name in feature_names[:5]:
    print(f"  {name}")

### 1.2 Discretise Features

Standard mutual information estimation requires discrete inputs. We use quantile-based binning (10 bins), which ensures equal frequency in each bin and is robust to skewed distributions.

**Why quantile binning?** Equal-frequency bins ensure each bin has enough samples for reliable MI estimation. Equal-width binning can leave bins nearly empty for skewed features, causing poor MI estimates.

In [ ]:
# Discretise all continuous features using quantile-based binning
# n_bins=10 gives a good balance between resolution and sample count per bin
N_BINS = 10

kbd = KBinsDiscretizer(n_bins=N_BINS, encode='ordinal', strategy='quantile')
X = kbd.fit_transform(X_raw).astype(int)
y_d = y.astype(int)  # Binary target (already discrete)

print(f"Discretised feature matrix shape: {X.shape}")
print(f"Feature value range: [{X.min()}, {X.max()}]")
print(f"Bins per feature: {N_BINS} (quantile-based)")
print(f"\nSample count per bin (feature 0): {np.bincount(X[:, 0])}")

---
## 2. Core MI Primitives

All five ITFS criteria reduce to combinations of three MI quantities:
1. Marginal MI: $I(x_k; y)$
2. Pairwise MI: $I(x_k; x_j)$
3. Conditional MI: $I(x_k; x_j \mid y)$ and $I(x_k; y \mid x_j)$

We implement these once and reuse them across all criteria.

In [ ]:
def mi(a, b):
    """Mutual information I(a; b) for discrete arrays."""
    return mutual_info_score(a, b)


def cmi_via_chain(a, b, c):
    """
    Conditional mutual information I(a; b | c).

    Uses the chain rule: I(a; b | c) = I(a; b, c) - I(a; c)
    We encode (b, c) as a single joint variable.

    This avoids iterating over values of c (which requires small partitions)
    and instead encodes the joint variable directly.
    """
    # Encode (b, c) as a joint integer variable
    n_c = int(np.max(c)) + 1
    bc = b * n_c + c  # Unique integer for each (b, c) pair
    # I(a; b | c) = I(a; b, c) - I(a; c)
    return max(0.0, mi(a, bc) - mi(a, c))


# Verify: I(a; b | c) >= 0 always
rng = np.random.default_rng(0)
a_test = rng.integers(0, 5, 200)
b_test = rng.integers(0, 5, 200)
c_test = rng.integers(0, 5, 200)
result = cmi_via_chain(a_test, b_test, c_test)
print(f"CMI test (random data, should be near 0): {result:.4f}")

# Verify: I(X; X | Z) is high when X and Z are independent
result2 = cmi_via_chain(a_test, a_test, c_test)
print(f"CMI test I(X; X | Z) (should be positive): {result2:.4f}")

---
## 3. The Five ITFS Criteria

We implement all five criteria as score functions with the same signature:
```python
criterion(x_k_idx, y, S_indices, X_data) -> float
```

Each returns a higher score for better candidates. The greedy selector calls the criterion at each step to pick the next feature.

In [ ]:
def score_mrmr(k, y, S, X):
    """
    mRMR: Maximum Relevance Minimum Redundancy (Peng et al., 2005)

    J = I(x_k; y) - (1/|S|) * sum_{x_j in S} I(x_k; x_j)

    CLM parameters: beta = 1/|S|, gamma = 0
    Weakness: ignores complementarity (gamma=0)
    """
    relevance = mi(X[:, k], y)
    if not S:
        return relevance
    redundancy = np.mean([mi(X[:, k], X[:, j]) for j in S])
    return relevance - redundancy


def score_jmi(k, y, S, X):
    """
    JMI: Joint Mutual Information (Yang & Moody, 1999)

    J = I(x_k; y) - (1/|S|) * sum_j I(x_k; x_j)
         + (1/|S|) * sum_j I(x_k; x_j | y)

    CLM parameters: beta = 1/|S|, gamma = 1/|S|
    This is the MLE of the CLM objective -- the principled default.
    """
    relevance = mi(X[:, k], y)
    if not S:
        return relevance
    redundancy = np.mean([mi(X[:, k], X[:, j]) for j in S])
    complement = np.mean([cmi_via_chain(X[:, k], X[:, j], y) for j in S])
    return relevance - redundancy + complement


def score_cmim(k, y, S, X):
    """
    CMIM: Conditional Mutual Information Maximisation (Fleuret, 2004)

    J = min_{x_j in S} I(x_k; y | x_j)

    Minimax criterion: guarantees information gain even given
    the worst-case already-selected feature.
    """
    if not S:
        return mi(X[:, k], y)
    cmis = [cmi_via_chain(X[:, k], y, X[:, j]) for j in S]
    return min(cmis)


def score_icap(k, y, S, X):
    """
    ICAP: Interaction Capping (Jakulin, 2005)

    J = I(x_k; y) - sum_j max(0, I(x_k; x_j) - I(x_k; x_j | y))

    Caps the penalty at zero: penalises redundancy but never
    rewards synergy. Conservative choice for noisy data.
    """
    relevance = mi(X[:, k], y)
    if not S:
        return relevance
    penalty = 0.0
    for j in S:
        redundancy = mi(X[:, k], X[:, j])
        complement = cmi_via_chain(X[:, k], X[:, j], y)
        # Interaction info: positive = redundant, negative = synergistic
        # ICAP caps the penalty: never reward synergy
        penalty += max(0.0, redundancy - complement)
    return relevance - penalty


def score_disr(k, y, S, X):
    """
    DISR: Double Input Symmetrical Relevance (Meyer et al., 2008)

    J = (1/|S|) * sum_j I(x_k, x_j; y) / H(x_k, x_j, y)

    Normalised JMI: scores in [0, 1], handles different cardinalities.
    """
    if not S:
        return mi(X[:, k], y)

    scores = []
    n = X.shape[0]
    for j in S:
        # Joint MI I(x_k, x_j; y): encode (x_k, x_j) as joint variable
        n_xj = int(np.max(X[:, j])) + 1
        xk_xj = X[:, k] * n_xj + X[:, j]
        joint_mi = mi(xk_xj, y)

        # Joint entropy H(x_k, x_j, y)
        n_y = int(np.max(y)) + 1
        xk_xj_y = xk_xj * n_y + y
        _, counts = np.unique(xk_xj_y, return_counts=True)
        probs = counts / n
        joint_entropy = -np.sum(probs * np.log(probs + 1e-12))

        if joint_entropy > 1e-10:
            scores.append(joint_mi / joint_entropy)
        else:
            scores.append(0.0)

    return np.mean(scores)


print("All five criteria implemented.")
print("Signature: score_*(k, y, S, X) -> float (higher = better candidate)")

---
## 4. Unified Greedy ITFS Selector

The Brown et al. framework shows that all five criteria plug into exactly the same greedy forward selection loop. The selector is the algorithm; the criterion is just a scoring function.

In [ ]:
def itfs_select(X, y, k, criterion_fn, feature_names=None, verbose=False):
    """
    Unified greedy ITFS selector.

    Selects k features by greedily maximising the criterion score
    at each step. All five criteria (JMI, CMIM, DISR, ICAP, mRMR)
    plug into this same loop.

    Parameters
    ----------
    X : array of shape (n, p)
        Discretised feature matrix (integer-valued)
    y : array of shape (n,)
        Discretised target variable
    k : int
        Number of features to select
    criterion_fn : callable
        Scoring function with signature (k_idx, y, S, X) -> float
    feature_names : list of str, optional
        Feature names for verbose output
    verbose : bool
        Print selection order if True

    Returns
    -------
    selected : list of int
        Indices of selected features in selection order
    """
    p = X.shape[1]
    selected = []       # Already-selected feature indices
    candidates = list(range(p))  # Remaining candidates

    for step in range(k):
        # Score all remaining candidates given the current selected set
        scores = {}
        for f in candidates:
            scores[f] = criterion_fn(f, y, selected, X)

        # Select the highest-scoring candidate
        best = max(scores, key=scores.__getitem__)
        selected.append(best)
        candidates.remove(best)

        if verbose:
            name = feature_names[best] if feature_names else str(best)
            print(f"  Step {step+1:2d}: {name:30s} (score={scores[best]:.4f})")

    return selected


# Quick sanity check: run mRMR for 3 features
print("mRMR — first 3 selections:")
test_sel = itfs_select(X, y_d, k=3, criterion_fn=score_mrmr,
                       feature_names=feature_names, verbose=True)

---
## 5. Run All Five Criteria

We select the top 10 features using each criterion and record both the selected indices and the wall-clock time. This takes ~30 seconds total.

In [ ]:
import time

K = 10  # Number of features to select

criteria = {
    'mRMR': score_mrmr,
    'JMI':  score_jmi,
    'CMIM': score_cmim,
    'ICAP': score_icap,
    'DISR': score_disr,
}

results = {}

for name, fn in criteria.items():
    t0 = time.time()
    selected = itfs_select(X, y_d, k=K, criterion_fn=fn)
    elapsed = time.time() - t0
    results[name] = {
        'selected': selected,
        'feature_names': [feature_names[i] for i in selected],
        'time_s': elapsed
    }
    print(f"{name:6s}: {elapsed:.2f}s | Features: {[feature_names[i][:15] for i in selected]}")

---
## 6. Overlap Analysis: Jaccard Similarity

The Jaccard similarity between two feature sets $A$ and $B$ measures how much they overlap:
$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

$J = 1$ means identical sets; $J = 0$ means completely disjoint. High Jaccard between criteria means they agree on which features are important.

In [ ]:
def jaccard_similarity(set_a, set_b):
    """Jaccard similarity between two sets: |A ∩ B| / |A ∪ B|"""
    a = set(set_a)
    b = set(set_b)
    intersection = len(a & b)
    union = len(a | b)
    return intersection / union if union > 0 else 1.0


criterion_names = list(results.keys())
n_crit = len(criterion_names)
jaccard_matrix = np.zeros((n_crit, n_crit))

for i, c1 in enumerate(criterion_names):
    for j, c2 in enumerate(criterion_names):
        jaccard_matrix[i, j] = jaccard_similarity(
            results[c1]['selected'],
            results[c2]['selected']
        )

# Display as DataFrame for readability
jaccard_df = pd.DataFrame(
    jaccard_matrix,
    index=criterion_names,
    columns=criterion_names
)
print("Jaccard Similarity Matrix (higher = more similar feature sets)")
print("=" * 55)
print(jaccard_df.round(3).to_string())

---
## 7. Visualise Feature Prioritisation

We create a binary selection matrix showing which criterion selected which feature. Features are sorted by the number of criteria that selected them, revealing consensus features (selected by all criteria) and disputed features (selected by only one).

In [ ]:
# Build selection matrix: rows = criteria, cols = features
all_features = list(range(len(feature_names)))
selection_matrix = np.zeros((n_crit, len(feature_names)), dtype=int)

for i, crit in enumerate(criterion_names):
    for f in results[crit]['selected']:
        selection_matrix[i, f] = 1

# Count how many criteria selected each feature
selection_counts = selection_matrix.sum(axis=0)

# Sort features by selection count (descending)
sort_order = np.argsort(-selection_counts)
sorted_names = [feature_names[i] for i in sort_order]
sorted_matrix = selection_matrix[:, sort_order]
sorted_counts = selection_counts[sort_order]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: binary selection heatmap
ax = axes[0]
im = ax.imshow(sorted_matrix, aspect='auto', cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(feature_names)))
ax.set_xticklabels([n[:12] for n in sorted_names],
                   rotation=60, ha='right', fontsize=7)
ax.set_yticks(range(n_crit))
ax.set_yticklabels(criterion_names, fontsize=10)
ax.set_title(f'Feature Selection Heatmap (Top {K} per Criterion)', fontsize=12)
ax.set_xlabel('Feature (sorted by consensus)', fontsize=10)

# Add text annotations
for i in range(n_crit):
    for j in range(len(feature_names)):
        if sorted_matrix[i, j] == 1:
            ax.text(j, i, 'x', ha='center', va='center',
                    color='white', fontsize=8, fontweight='bold')

# Right: Jaccard similarity heatmap
ax2 = axes[1]
im2 = ax2.imshow(jaccard_matrix, cmap='RdYlGn', vmin=0, vmax=1)
ax2.set_xticks(range(n_crit))
ax2.set_xticklabels(criterion_names, fontsize=10)
ax2.set_yticks(range(n_crit))
ax2.set_yticklabels(criterion_names, fontsize=10)
ax2.set_title('Jaccard Similarity Between Criteria', fontsize=12)
plt.colorbar(im2, ax=ax2)

for i in range(n_crit):
    for j in range(n_crit):
        ax2.text(j, i, f'{jaccard_matrix[i, j]:.2f}',
                 ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('feature_selection_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

# Print consensus analysis
print("\nConsensus Analysis:")
print(f"  Selected by all 5 criteria: "
      f"{[feature_names[sort_order[i]] for i in range(len(feature_names)) if sorted_counts[i] == 5]}")
print(f"  Selected by exactly 1 criterion: "
      f"{sum(sorted_counts == 1)} features")

---
## 8. Downstream Model Performance

The ultimate test: do different ITFS criteria lead to different downstream accuracy? We evaluate each selected feature set using 5-fold cross-validated Random Forest accuracy.

In [ ]:
# Evaluate each criterion's selected features using Random Forest
# Use raw (non-discretised) features for the model — discretisation was only for MI
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

performance = {}

# Baseline: all 30 features
baseline_scores = cross_val_score(clf, X_scaled, y, cv=cv, scoring='accuracy')
performance['Baseline (all 30)'] = {
    'mean': baseline_scores.mean(),
    'std': baseline_scores.std(),
    'n_features': X_scaled.shape[1]
}

# Each ITFS criterion
for crit_name, res in results.items():
    X_sel = X_scaled[:, res['selected']]
    scores = cross_val_score(clf, X_sel, y, cv=cv, scoring='accuracy')
    performance[crit_name] = {
        'mean': scores.mean(),
        'std': scores.std(),
        'n_features': K
    }

# Print results table
print(f"{'Method':<22} {'Features':>8} {'CV Accuracy':>12} {'Std':>8}")
print("-" * 55)
for method, perf in performance.items():
    marker = "*" if perf['mean'] == max(p['mean'] for p in performance.values()) else " "
    print(f"{method:<22} {perf['n_features']:>8} "
          f"{perf['mean']:>11.4f} {perf['std']:>8.4f} {marker}")
print("* = best")

### Visualise Performance vs. Feature Count

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: bar chart of accuracy with error bars
ax = axes[0]
methods = list(performance.keys())
means = [performance[m]['mean'] for m in methods]
stds = [performance[m]['std'] for m in methods]
colors = ['gray'] + ['steelblue', 'darkorange', 'green', 'red', 'purple']

bars = ax.barh(methods, means, xerr=stds, color=colors, alpha=0.8,
               error_kw={'capsize': 4, 'ecolor': 'black'})
ax.axvline(x=means[0], color='gray', linestyle='--', alpha=0.5,
           label='Baseline (all features)')
ax.set_xlabel('5-Fold CV Accuracy', fontsize=11)
ax.set_title(f'Downstream Accuracy ({K} features vs. all)', fontsize=12)
ax.set_xlim([0.9, 1.0])
ax.legend(fontsize=9)

# Add accuracy labels
for i, (mean, std) in enumerate(zip(means, stds)):
    ax.text(mean + std + 0.001, i, f'{mean:.4f}', va='center', fontsize=8)

# Right: runtime comparison
ax2 = axes[1]
crit_only = [c for c in methods if c != 'Baseline (all 30)']
times = [results[c]['time_s'] for c in crit_only]
crit_colors = ['steelblue', 'darkorange', 'green', 'red', 'purple']
ax2.bar(crit_only, times, color=crit_colors, alpha=0.8)
ax2.set_ylabel('Runtime (seconds)', fontsize=11)
ax2.set_title('Computational Cost per Criterion', fontsize=12)
ax2.set_xlabel('Criterion', fontsize=11)
for i, t in enumerate(times):
    ax2.text(i, t + 0.01, f'{t:.2f}s', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('itfs_performance_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 9. Exercise: Modify the CLM Parameters

The Brown et al. (2012) framework shows that all criteria are special cases of:
$$J(x_k) = I(x_k; y) + \sum_{x_j \in S} [\gamma_j \cdot I(x_k; x_j | y) - \beta_j \cdot I(x_k; x_j)]$$

**Task:** Implement a **parametric CLM criterion** where $\beta$ and $\gamma$ are free parameters. Verify that:
- $\beta = 1/|S|$, $\gamma = 0$ recovers mRMR
- $\beta = 1/|S|$, $\gamma = 1/|S|$ recovers JMI

Then find the $(\beta, \gamma)$ pair that maximises cross-validated accuracy on this dataset.

In [ ]:
# YOUR CODE HERE
# --------------------------------------------------
def make_clm_criterion(beta_scale, gamma_scale):
    """
    Factory function: returns a CLM criterion with given beta and gamma scale factors.

    The final beta_j = beta_scale / |S| and gamma_j = gamma_scale / |S|
    (so setting beta_scale=1, gamma_scale=0 gives mRMR;
         beta_scale=1, gamma_scale=1 gives JMI)

    Parameters
    ----------
    beta_scale : float
        Scaling factor for redundancy penalty
    gamma_scale : float
        Scaling factor for complementarity reward

    Returns
    -------
    callable : criterion function with signature (k, y, S, X) -> float
    """
    # TODO: implement this function
    # Hint: copy score_jmi and replace 1/|S| with beta_scale/|S| and gamma_scale/|S|
    pass


# TODO: Verify recovery of mRMR (beta_scale=1, gamma_scale=0)
# clm_mrmr = make_clm_criterion(beta_scale=1.0, gamma_scale=0.0)
# sel_clm_mrmr = itfs_select(X, y_d, k=K, criterion_fn=clm_mrmr)
# print("CLM recovers mRMR:", sorted(sel_clm_mrmr) == sorted(results['mRMR']['selected']))

# TODO: Verify recovery of JMI (beta_scale=1, gamma_scale=1)

# TODO: Grid search over beta_scale in [0.5, 1.0, 1.5] and
#         gamma_scale in [0.0, 0.5, 1.0, 1.5]
# Evaluate CV accuracy for each and find the best combination

In [ ]:
# AUTO-GRADED TEST
# --------------------------------------------------
def test_clm_criterion():
    assert make_clm_criterion is not None, "Implement make_clm_criterion"

    fn = make_clm_criterion(1.0, 0.0)  # Should behave like mRMR
    assert fn is not None, "make_clm_criterion should return a callable"
    assert callable(fn), "Return value must be callable"

    # Score a single feature (no selected set)
    score = fn(0, y_d, [], X)
    assert isinstance(score, (float, np.floating)), \
        f"Score must be a float, got {type(score)}"
    assert score >= 0, "MI is non-negative, so score with empty S must be >= 0"

    print("Test passed: make_clm_criterion returns a valid callable.")
    print(f"Test score for feature 0 (empty S, mRMR-like): {score:.4f}")
    print(f"Compare to mRMR score: {score_mrmr(0, y_d, [], X):.4f}")


# Uncomment to run after implementing:
# test_clm_criterion()

---
## 10. Summary

### Key Takeaways

1. **All five criteria are one algorithm** — they differ only in $\beta_j$ and $\gamma_j$ in the CLM objective from Brown et al. (2012)

2. **JMI is the principled default** — it maximises the expected naive Bayes log-likelihood and consistently performs well empirically

3. **mRMR's weakness is $\gamma = 0$** — it ignores complementarity and performs worst when synergistic features exist

4. **Jaccard similarity reveals practical agreement** — on well-behaved datasets, most criteria agree on the top features; disagreements reveal where the choice of criterion matters most

5. **Downstream accuracy is the final arbiter** — use 5-fold CV to validate which criterion actually helps your model

### What's Next

- **Notebook 02**: Transfer entropy — when features drive the target in time, not just correlate with it
- **Guide 02**: Advanced measures — Rényi MI, copula dependence, interaction information
- **Module 03**: Wrapper methods — replace MI with actual model performance

### References

- Brown, G. et al. (2012). Conditional Likelihood Maximisation. *JMLR* 13, 27–66.
- Peng, H. et al. (2005). Feature selection based on mutual information. *IEEE TPAMI*.
- Fleuret, F. (2004). Fast binary feature selection with conditional MI. *JMLR* 5.